In [63]:
import json
import glob
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, BertForSequenceClassification

model_dir = "./FinBERT-PT-BR"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = BertForSequenceClassification.from_pretrained(model_dir)
model.eval()

LABEL_MAP = {0: "POSITIVE", 1: "NEGATIVE", 2: "NEUTRAL"}

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Device: {device}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8601.73it/s]
BertForSequenceClassification LOAD REPORT from: ./FinBERT-PT-BR
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda


In [64]:
def predict_batch(texts, batch_size=32):
    """Predict sentiment for a list of texts in batches."""
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        tokens = tokenizer(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=512).to(device)
        with torch.no_grad():
            logits = model(**tokens).logits.cpu().numpy()
        for j, log in enumerate(logits):
            cls = int(np.argmax(log))
            results.append({
                "sentiment_class": cls,
                "sentiment": LABEL_MAP[cls],
                "logits": log.tolist(),
            })
    return results

In [65]:
# Load all news files
news_files = sorted(glob.glob("../1.news/*_noticias.json"))
print(f"Found {len(news_files)} news files: {[f.split('/')[-1] for f in news_files]}")

all_results = {}

for fpath in news_files:
    ticker = fpath.split("/")[-1].replace("_noticias.json", "").upper()
    with open(fpath, "r") as f:
        articles = json.load(f)

    print(f"\n{'='*50}")
    print(f"{ticker}: {len(articles)} articles")

    # Use title + excerpt as input (more focused than full content)
    texts = [f"{a['title']}. {a['excerpt']}" for a in articles]

    preds = predict_batch(texts, batch_size=32)

    for article, pred in zip(articles, preds):
        article["sentiment"] = pred["sentiment"]
        article["sentiment_class"] = pred["sentiment_class"]
        article["sentiment_logits"] = pred["logits"]

    # Save enriched JSON
    out_path = f"{ticker.lower()}_noticias_sentiment.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(articles, f, ensure_ascii=False, indent=2)

    # Summary stats
    sentiments = [p["sentiment"] for p in preds]
    print(f"  POSITIVE: {sentiments.count('POSITIVE')} ({sentiments.count('POSITIVE')/len(sentiments)*100:.1f}%)")
    print(f"  NEGATIVE: {sentiments.count('NEGATIVE')} ({sentiments.count('NEGATIVE')/len(sentiments)*100:.1f}%)")
    print(f"  NEUTRAL:  {sentiments.count('NEUTRAL')} ({sentiments.count('NEUTRAL')/len(sentiments)*100:.1f}%)")
    print(f"  Saved to: {out_path}")

    all_results[ticker] = articles

Found 3 news files: ['itub4_noticias.json', 'petr4_noticias.json', 'vale3_noticias.json']

ITUB4: 2572 articles
  POSITIVE: 756 (29.4%)
  NEGATIVE: 1107 (43.0%)
  NEUTRAL:  709 (27.6%)
  Saved to: itub4_noticias_sentiment.json

PETR4: 7758 articles
  POSITIVE: 1756 (22.6%)
  NEGATIVE: 3594 (46.3%)
  NEUTRAL:  2408 (31.0%)
  Saved to: petr4_noticias_sentiment.json

VALE3: 5134 articles
  POSITIVE: 1400 (27.3%)
  NEGATIVE: 2417 (47.1%)
  NEUTRAL:  1317 (25.7%)
  Saved to: vale3_noticias_sentiment.json


In [66]:
# Build daily sentiment DataFrame per ticker (for merging with price data)
for ticker, articles in all_results.items():
    df = pd.DataFrame([{
        "date": a["date"][:10],
        "sentiment_class": a["sentiment_class"],
        "logit_pos": a["sentiment_logits"][0],
        "logit_neg": a["sentiment_logits"][1],
        "logit_neu": a["sentiment_logits"][2],
    } for a in articles])

    df["date"] = pd.to_datetime(df["date"])

    # Aggregate per day: mean logits + count + dominant sentiment
    daily = df.groupby("date").agg(
        n_articles=("sentiment_class", "count"),
        mean_logit_pos=("logit_pos", "mean"),
        mean_logit_neg=("logit_neg", "mean"),
        mean_logit_neu=("logit_neu", "mean"),
        mean_sentiment=("sentiment_class", "mean"),
    ).sort_index()

    out_csv = f"{ticker.lower()}_daily_sentiment.csv"
    daily.to_csv(out_csv)
    print(f"{ticker}: {len(daily)} days → {out_csv}")
    display(daily.tail(10))

ITUB4: 1115 days → itub4_daily_sentiment.csv


,n_articles,mean_logit_pos,mean_logit_neg,mean_logit_neu,mean_sentiment
date,,,,,
2026-03-02,3,-1.132515,0.794860,-1.726048,0.666667
2026-03-03,2,-1.479831,-0.047439,-0.450596,1.500000
2026-03-04,4,-0.892064,-1.034987,-0.299213,1.250000
2026-03-05,2,-1.749076,1.751523,-1.750448,1.000000
2026-03-06,2,-1.301058,0.111944,-0.921083,1.500000
2026-03-09,3,0.313861,-1.455899,-1.134381,0.666667
2026-03-10,2,-0.451034,-0.131313,-1.623372,0.500000
2026-03-11,2,-0.178097,-1.352347,-0.956858,0.500000
2026-03-13,5,-0.828365,0.109605,-1.370350,0.600000


PETR4: 1474 days → petr4_daily_sentiment.csv


,n_articles,mean_logit_pos,mean_logit_neg,mean_logit_neu,mean_sentiment
date,,,,,
2026-03-04,7,-1.017022,-0.029959,-1.093917,0.857143
2026-03-05,14,-1.493921,0.249268,-0.802366,1.285714
2026-03-06,21,-0.715722,-0.603599,-0.706900,1.095238
2026-03-07,1,-1.088093,-1.713217,0.739551,2.000000
2026-03-09,17,-1.127509,-0.232810,-0.751045,1.176471
2026-03-10,9,-1.461400,0.257276,-1.048939,1.111111
2026-03-11,10,-0.857091,-0.930786,-0.488834,1.100000
2026-03-12,9,-1.727117,0.540078,-0.907505,1.333333
2026-03-13,14,-1.327671,0.355492,-1.150138,1.000000


VALE3: 1354 days → vale3_daily_sentiment.csv


,n_articles,mean_logit_pos,mean_logit_neg,mean_logit_neu,mean_sentiment
date,,,,,
2026-03-03,6,-1.356260,0.053829,-0.616887,1.166667
2026-03-04,7,-1.020747,-0.426613,-0.787685,1.000000
2026-03-05,9,-1.273999,0.294710,-1.030661,1.111111
2026-03-06,8,-1.116726,-0.339996,-0.642647,1.375000
2026-03-09,5,-0.491440,-0.766163,-0.932994,1.000000
2026-03-10,3,-0.850716,-0.479739,-0.827744,1.000000
2026-03-11,11,-1.218505,-0.526455,-0.529208,1.363636
2026-03-12,6,-1.152518,0.387957,-1.285392,1.000000
2026-03-13,9,-1.029450,0.265664,-1.282685,0.888889
